In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

currentRoot = Path.cwd().resolve()
projectRoot = currentRoot.parent if currentRoot.name == 'notebooks' else currentRoot
if not (projectRoot / 'notebooks').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory.')

dataRoot = projectRoot / 'data'
rawRoot = dataRoot / 'raw'
interimRoot = dataRoot / 'interim'
processedRoot = dataRoot / 'processed'
figureRoot = projectRoot / 'reports' / 'figures'
for outputRoot in (rawRoot, interimRoot, processedRoot, figureRoot):
    outputRoot.mkdir(parents=True, exist_ok=True)

configuredRawRoot = os.environ.get('SKIN_DATA_ROOT')
rawCandidates = [Path(configuredRawRoot).expanduser()] if configuredRawRoot else []
rawCandidates.extend([rawRoot, projectRoot.parent / 'Data'])

def containsDatasetFiles(path: Path) -> bool:
    datasetSuffixes = {'.csv', '.zip', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    return path.exists() and any(
        candidate.is_file() and candidate.suffix.lower() in datasetSuffixes
        for candidate in path.rglob('*')
    )

sourceDataRoot = next((path.resolve() for path in rawCandidates if containsDatasetFiles(path)), None)
if sourceDataRoot is None:
    raise FileNotFoundError('No raw dataset found. Populate data/raw or set SKIN_DATA_ROOT.')

def toProjectPath(path: Path) -> str:
    return Path(os.path.relpath(path.resolve(), projectRoot)).as_posix()

def fromProjectPath(pathValue: str) -> Path:
    return (projectRoot / Path(pathValue)).resolve()

print(f'Project root: {projectRoot}')
print(f'Raw dataset root: {sourceDataRoot}')

Project root: D:\sunib\Research\Skin\skin-domain-shift
Raw dataset root: D:\sunib\Research\Skin\Data


In [2]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from tqdm.auto import tqdm

clinicalInventoryPath = interimRoot / 'clinicalInventory.csv'
isicInventoryPath = interimRoot / 'isicInventory.csv'
for requiredPath in (clinicalInventoryPath, isicInventoryPath):
    if not requiredPath.exists():
        raise FileNotFoundError(f'Run notebook 01 first; missing {requiredPath.name}.')

clinicalInventory = pd.read_csv(clinicalInventoryPath, dtype='string')
isicInventory = pd.read_csv(isicInventoryPath, dtype='string')

In [3]:
def validateImageRecord(record: dict) -> dict:
    imagePathValue = record.get('imagePath')
    result = {
        'imageId': record.get('imageId'),
        'imagePath': imagePathValue,
        'imageReadable': False,
        'imageWidth': pd.NA,
        'imageHeight': pd.NA,
        'imageFormat': pd.NA,
        'colorMode': pd.NA,
        'fileSize': pd.NA,
        'modifiedNs': pd.NA,
        'validationError': pd.NA,
    }
    if pd.isna(imagePathValue) or not str(imagePathValue).strip():
        result['validationError'] = 'Missing image path'
        return result
    imagePath = fromProjectPath(str(imagePathValue))
    if not imagePath.exists():
        result['validationError'] = 'Image file does not exist'
        return result
    try:
        with Image.open(imagePath) as image:
            image.verify()
        with Image.open(imagePath) as image:
            result.update({
                'imageReadable': True,
                'imageWidth': int(image.width),
                'imageHeight': int(image.height),
                'imageFormat': image.format,
                'colorMode': image.mode,
                'fileSize': imagePath.stat().st_size,
                'modifiedNs': imagePath.stat().st_mtime_ns,
            })
    except Exception as error:
        result['validationError'] = f'{type(error).__name__}: {error}'
    return result

def validateManifestImages(manifest: pd.DataFrame, workerCount: int = 8) -> pd.DataFrame:
    records = manifest[['imageId', 'imagePath']].to_dict('records')
    with ThreadPoolExecutor(max_workers=workerCount) as executor:
        results = list(tqdm(executor.map(validateImageRecord, records), total=len(records), desc='Validating images'))
    validationFrame = pd.DataFrame(results)
    return manifest.merge(validationFrame, on=['imageId', 'imagePath'], how='left', validate='one_to_one')

clinicalValidated = validateManifestImages(clinicalInventory)
isicValidated = validateManifestImages(isicInventory)
print(f'Clinical readable: {int(clinicalValidated.imageReadable.fillna(False).sum()):,}/{len(clinicalValidated):,}')
print(f'ISIC readable: {int(isicValidated.imageReadable.fillna(False).sum()):,}/{len(isicValidated):,}')

Validating images:   0%|          | 0/2298 [00:00<?, ?it/s]

Validating images:   0%|          | 0/33569 [00:00<?, ?it/s]

Clinical readable: 2,298/2,298
ISIC readable: 33,569/33,569


In [4]:
clinicalAllowedLabels = {'ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK'}
isicAllowedLabels = {'AK', 'BCC', 'MEL', 'NV', 'SCC', 'BKL', 'DF', 'VASC', 'UNK'}

def addValidationFlags(manifest: pd.DataFrame, datasetName: str) -> pd.DataFrame:
    result = manifest.copy()
    result['missingImageId'] = result['imageId'].isna() | result['imageId'].str.strip().eq('')
    result['duplicateImageId'] = result['imageId'].duplicated(keep=False)
    result['missingPatientId'] = (
        result['patientId'].isna() | result['patientId'].str.strip().eq('')
        if datasetName == 'clinical' else False
    )
    allowedLabels = clinicalAllowedLabels if datasetName == 'clinical' else isicAllowedLabels
    labelsPresent = result['originalLabel'].notna()
    result['invalidDiagnosisLabel'] = labelsPresent & ~result['originalLabel'].isin(allowedLabels)
    result['invalidImage'] = ~result['imageReadable'].fillna(False).astype(bool)
    result['isValid'] = ~result[
        ['missingImageId', 'duplicateImageId', 'missingPatientId', 'invalidDiagnosisLabel', 'invalidImage']
    ].any(axis=1)
    return result

clinicalValidated = addValidationFlags(clinicalValidated, 'clinical')
isicValidated = addValidationFlags(isicValidated, 'isic')

clinicalValidatedPath = interimRoot / 'clinicalValidated.csv'
isicValidatedPath = interimRoot / 'isicValidated.csv'
clinicalValidated.to_csv(clinicalValidatedPath, index=False)
isicValidated.to_csv(isicValidatedPath, index=False)

flagColumns = ['missingImageId', 'duplicateImageId', 'missingPatientId', 'invalidDiagnosisLabel', 'invalidImage']
invalidFrames = []
for datasetName, frame in [('clinical', clinicalValidated), ('isic2019', isicValidated)]:
    invalidFrame = frame.loc[~frame['isValid'].astype(bool), ['imageId', 'imagePath', 'validationError', *flagColumns]].copy()
    invalidFrame.insert(0, 'sourceDataset', datasetName)
    invalidFrame['invalidReasons'] = invalidFrame[flagColumns].apply(
        lambda row: '; '.join(column for column, value in row.items() if bool(value)), axis=1
    )
    invalidFrames.append(invalidFrame)
invalidSamples = pd.concat(invalidFrames, ignore_index=True)
invalidSamplesPath = interimRoot / 'invalidSamples.csv'
invalidSamples.to_csv(invalidSamplesPath, index=False)
invalidSamples.head()

,sourceDataset,imageId,imagePath,validationError,missingImageId,duplicateImageId,missingPatientId,invalidDiagnosisLabel,invalidImage,invalidReasons


In [5]:
def missingPercentages(frame: pd.DataFrame) -> dict[str, float]:
    return {
        column: round(float(frame[column].isna().mean() * 100), 4)
        for column in frame.columns
    }

def summarizeValidation(frame: pd.DataFrame) -> dict:
    return {
        'rowCount': len(frame),
        'validCount': int(frame['isValid'].astype(bool).sum()),
        'invalidCount': int((~frame['isValid'].astype(bool)).sum()),
        'readableImageCount': int(frame['imageReadable'].astype(bool).sum()),
        'duplicateImageIdCount': int(frame['duplicateImageId'].astype(bool).sum()),
        'missingImageIdCount': int(frame['missingImageId'].astype(bool).sum()),
        'missingPatientIdCount': int(frame['missingPatientId'].astype(bool).sum()),
        'invalidDiagnosisCount': int(frame['invalidDiagnosisLabel'].astype(bool).sum()),
        'formats': frame['imageFormat'].value_counts(dropna=False).to_dict(),
        'colorModes': frame['colorMode'].value_counts(dropna=False).to_dict(),
        'missingPercentages': missingPercentages(frame),
    }

validationReport = {
    'clinical': summarizeValidation(clinicalValidated),
    'isic': summarizeValidation(isicValidated),
    'isicGroundTruthAvailable': bool(isicValidated['originalLabel'].notna().any()),
    'invalidSamplesCount': len(invalidSamples),
}
validationReportPath = interimRoot / 'validationReport.json'
validationReportPath.write_text(json.dumps(validationReport, indent=2), encoding='utf-8')
print(json.dumps({
    'clinical': {key: value for key, value in validationReport['clinical'].items() if key != 'missingPercentages'},
    'isic': {key: value for key, value in validationReport['isic'].items() if key != 'missingPercentages'},
}, indent=2))

{
  "clinical": {
    "rowCount": 2298,
    "validCount": 2298,
    "invalidCount": 0,
    "readableImageCount": 2298,
    "duplicateImageIdCount": 0,
    "missingImageIdCount": 0,
    "missingPatientIdCount": 0,
    "invalidDiagnosisCount": 0,
    "formats": {
      "PNG": 2298
    },
    "colorModes": {
      "RGBA": 1440,
      "RGB": 858
    }
  },
  "isic": {
    "rowCount": 33569,
    "validCount": 33569,
    "invalidCount": 0,
    "readableImageCount": 33569,
    "duplicateImageIdCount": 0,
    "missingImageIdCount": 0,
    "missingPatientIdCount": 0,
    "invalidDiagnosisCount": 0,
    "formats": {
      "JPEG": 33569
    },
    "colorModes": {
      "RGB": 33569
    }
  }
}


In [6]:
import tempfile

with tempfile.TemporaryDirectory(dir=interimRoot) as temporaryName:
    temporaryRoot = Path(temporaryName)
    validPath = temporaryRoot / 'valid.png'
    brokenPath = temporaryRoot / 'broken.png'
    Image.new('RGB', (13, 17), color=(20, 40, 60)).save(validPath)
    brokenPath.write_bytes(b'not-an-image')
    validResult = validateImageRecord({'imageId': 'valid', 'imagePath': toProjectPath(validPath)})
    brokenResult = validateImageRecord({'imageId': 'broken', 'imagePath': toProjectPath(brokenPath)})
    assert validResult['imageReadable'] and validResult['imageWidth'] == 13 and validResult['imageHeight'] == 17
    assert not brokenResult['imageReadable'] and pd.notna(brokenResult['validationError'])
print('Synthetic image validation checks passed.')

Synthetic image validation checks passed.
